# MNIST Neural Network

In [72]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Importing Dataset

In [73]:
data = pd.read_csv("train.csv")

In [74]:
data.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Formatting Data

In [75]:
data = np.array(data)

#Getting dimensions of the data
#42000 rows, 785 cols -> 1 label + 784 pixels
m,n = data.shape

#For training shuffling the data
np.random.shuffle(data)

In [76]:
print(m,n)

42000 785


### Important Terminologies
We use separate datasets to learn, tune, and evaluate a model.

**Train set:** 
- The model learns digit patterns by updating weights using backpropagation
   
**Dev (validation) set:**
- Used to tune hyperparameters and training decisions
  
**Test set:**
- Used once at the end for final performance measurement.
- Here the labels are hidden, so Kaggle evaluates the predictions for us

This separation prevents overfitting and ensures reliable generalization.

So just a dev/train split is needed

In [77]:
#Creating dev/train split to check for overfitting
#Following 80% train 20% dev split
split = int(0.2 * m)
data_dev = data[0:split]

#Transposing the data
data_dev = data_dev.T
#Label values
Y_dev = data_dev[0]
#Remove the label column only pixels
X_dev = data_dev[1:n]

In [78]:
data_train = data[split:m]

#Transposing the data
data_train = data_train.T
#Label values
Y_train = data_train[0]
#Remove the label column only pixels
X_train = data_train[1:n]

### X Y split explained
- After transposing the numpy array,
- label became the first row,
- the rest 1:785 columns are pixels

- Thus, data_(dev/train)[0] gives lables
- And, data_(dev/train)[1:785] gives pixels
- Visualizing for better understanding

In [79]:
data_dev #First row is labels

array([[7, 8, 4, ..., 1, 1, 4],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(785, 8400))

In [80]:
Y_dev

array([7, 8, 4, ..., 1, 1, 4], shape=(8400,))

In [81]:
X_dev

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(784, 8400))

In [82]:
data_train #First row is labels

array([[5, 2, 3, ..., 8, 2, 6],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(785, 33600))

In [83]:
Y_train

array([5, 2, 3, ..., 8, 2, 6], shape=(33600,))

In [84]:
X_train

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(784, 33600))

# Neural Net

## Maths behind initializing parameters
- We need to generate random values for the weights and biases for initialization
- So what range to keep to ensure
- No vanishing or exploding gradients/activations
- Control parameters needed:
    - mean = 0
    - controlled variance

### Why mean should be 0?
Let us look at once neuron
  $$z = w_1x_1 + w_2x_2 + \dots + w_nx_n$$
Taking expectation or average behaviour
  $$E[z] = E[w_1]E[x_1] + \dots + E[w_n]E[x_n]$$
If weights are symmetric around 0, that is mean = 0
  $$E[w_i] = 0 \Rightarrow E[z] = 0$$
Thus half neurons positive, half neurons negative

Thus we have a variety in gradients

If mean ≠ 0
  $$E[w_i] \neq 0 \Rightarrow z \text{ shifts in one direction}$$

### Why controlled variance is needed?
Controlled variance implies that we can't have a big range
- Variance of weighted sum:
    $$\text{Var}(z) = n \cdot \text{Var}(w) \cdot \text{Var}(x)$$
- In each layer we will be multipliying variance
  $$\text{Var}(z_L) = k^L \text{Var}(z_0)$$
- So after a few layers we will have $$z \to \infty$$

This is the **vanishing/exploding gradient problem.**

### Why (-0.5, 0.5) Range Chosen?
- Mean = 0
- Small Range
- But also it satisfies the condition:
  $$\text{Var}(w) = \frac{2}{n_{in}} \text{ (ReLU)}$$
- Because we want
  $$\text{Var}(z_{\text{layer}}) \approx \text{Var}(z_{\text{previous}})$$
- Proof :
  - In each layer we multiply gradients, assuming k layers
    $$\text{Var}(z_L) = k^L \text{Var}(z_0)$$
  - We want signal strength to be preserved, that is not vanish or explode, So
    $$\text{Var}(z_{\text{layer}}) \approx \text{Var}(z_{\text{previous}})$$
  - Following constraints need to be met
    | Condition | Outcome |
| :--- | :--- |
| **$k > 1$** | **Explosion:** Variance grows exponentially, leading to numerical instability. |
| **$k < 1$** | **Vanishing:** Variance shrinks to zero; the network stops learning. |
| **$k = 1$** | **Stable Learning:** Signal variance is preserved throughout the network. |
  - Thus, solving for weight variance, we get
    $$\text{Var}(w) = \frac{2}{n_{in}} \text{ (ReLU)}$$
  - This gives **He Initialization**
    $$W \sim \mathcal{N}\left(0, \frac{2}{n_{in}}\right)$$

### Range Analysis
| Range | What happens |
| :--- | :--- |
| **$(0, 1)$** | **Neurons identical:** Symmetry isn't broken; neurons learn the same features. |
| **$(-1, 0)$** | **Neurons dead:** For ReLU, weights start in the "off" zone, causing dead neurons, also known as **dying ReLU problem** |
| **$(-1, 1)$** | **Exploding signals:** The range is too wide for deep architectures without normalization. |
| **$(-0.5, 0.5)$** | **Works shallow only:** Initial signal is okay, but degrades in deeper layers. |
| **He Init** | **Stable deep learning:** Specifically tuned to keep $k \approx 1$ for ReLU. |

## How to initialize parameters
- For input layer , or Layer 0,
- We have 784 neurons, 1st layer has 32 neurons
- Now, each neuron in 0th layer is connected to each neuron in 1st layer

### Weights
- so for that 1 singular neuron we need 32 weights for each of it's connections in the 1st layer
- Similary we have to do it for all the neurons in the 0th layer, which are 784 in total
- So we can generate a random values array of size :
- 32 rows x 784 columns

### Bias
- For each singular neuron in 1st layer, we need to add a singular bias value
- Thus in 1st layer we have 32 neurons so, we need 32 bias values
- So we can generate a random values array of size :
- 32 rows x 1 column

### Layers
- Now for each layer we can repeat this process

### Generalizing
- **Wn = np.random.rand(len(layer n+1), len(layer n))**
- **Bn = np.random.rand(len(layer n+1), 1)**

### Architecture
layers(784,32,16,10)
- 784 neurons in input layer
- 10 neurons in output layer(for each digit 0-9)
- 2 hidden layers
- 1st hidden layer is 32 neurons
- 2nd hidden layer is 16 neurons

In [89]:
#Initializing the starting weights and biases
#np.random.rand gives value between [0,1] so,
#[0,1] - 0.5 = [-0.5,0.5]
def init_params():
    W1 = np.random.rand(32,784) - 0.5
    B1 = np.random.rand(32,1) - 0.5
    W2 = np.random.rand(16,32) - 0.5
    B2 = np.random.rand(16,1) - 0.5
    W3 = np.random.rand(10,16) - 0.5
    B3 = np.random.rand(10,1) - 0.5
    return W1,B1,W2,B2,W3,B3    